# Leading, Coincident & Lagging Macro Indicators

Business-cycle indicators are conventionally grouped by *when* they move relative to the cycle, following the Conference Board's classification:

- **Leading** — turn before the cycle (yield curve, jobless claims, building permits, new orders, consumer sentiment, manufacturing hours).
- **Coincident** — move with the cycle (payrolls, industrial production, personal income, retail sales).
- **Lagging** — confirm the cycle after the fact (unemployment duration, business loans, the prime rate, core CPI, unit labor cost).

This notebook queries 16 US indicators from FRED (all live-verified 2026-07-16) plus a deliberately thin 3-indicator Thailand panel (SET Index, GDP growth, CPI inflation — the only keyless, reasonably current sources available for this framework), and plots each US category as a normalized trend shaded by NBER recession periods so the lead/coincident/lag relationship is visible directly in the chart.

**Standalone notebook:** series are fetched directly via `FredSource`/`WorldBankSource`/`yfinance`, not through `macro_data.update()`/`catalog.yaml` — nothing here is persisted to `data/`. See `macro_cycle_indicators_catalog.csv` alongside this notebook for the full roster with tags.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from macro_data.catalog import SeriesConfig
from macro_data.sources.fred import FredSource
from macro_data.sources.worldbank import WorldBankSource

pd.set_option("display.max_rows", 25)
%matplotlib inline

In [2]:
roster = pd.read_csv("macro_cycle_indicators_catalog.csv")
roster

,id,region,source,tag,frequency,name
0,T10Y3M,US,fred,leading,daily,10Y-3M Treasury yield spread
1,ICSA,US,fred,leading,weekly,Initial jobless claims
2,PERMIT,US,fred,leading,monthly,Building permits (new private housing units au...
3,NEWORDER,US,fred,leading,monthly,Manufacturers' new orders: core capital goods ...
4,UMCSENT,US,fred,leading,monthly,U. Michigan Consumer Sentiment Index
5,AWHMAN,US,fred,leading,monthly,Average weekly hours (manufacturing)
6,PAYEMS,US,fred,coincident,monthly,Nonfarm payroll employment
7,INDPRO,US,fred,coincident,monthly,Industrial production index
8,W875RX1,US,fred,coincident,monthly,Real personal income excluding transfer receipts
9,USPHCI,US,fred,coincident,monthly,Coincident Economic Activity Index (composite)


In [3]:
FRED_SERIES = {
    "T10Y3M": ("leading", "D"),
    "ICSA": ("leading", "W"),
    "PERMIT": ("leading", "M"),
    "NEWORDER": ("leading", "M"),
    "UMCSENT": ("leading", "M"),
    "AWHMAN": ("leading", "M"),
    "PAYEMS": ("coincident", "M"),
    "INDPRO": ("coincident", "M"),
    "W875RX1": ("coincident", "M"),
    "USPHCI": ("coincident", "M"),
    "RSAFS": ("coincident", "M"),
    "UEMPMEAN": ("lagging", "M"),
    "BUSLOANS": ("lagging", "M"),
    "MPRIME": ("lagging", "M"),
    "CPILFESL": ("lagging", "M"),
    "ULCNFB": ("lagging", "Q"),
}

fred = FredSource()
us_raw = {}
us_failed = {}
for series_id, (tag, freq) in FRED_SERIES.items():
    cfg = SeriesConfig(id=series_id, source="fred", name=series_id, params={"series": series_id})
    try:
        us_raw[series_id] = fred.fetch(cfg)
    except Exception as exc:
        us_failed[series_id] = str(exc)
        print(f"failed to fetch {series_id}: {exc}")

# NBER recession indicator — used only for chart shading, not a tagged series itself
recession_cfg = SeriesConfig(id="USREC", source="fred", name="USREC", params={"series": "USREC"})
usrec = fred.fetch(recession_cfg)

print(f"fetched {len(us_raw)}/{len(FRED_SERIES)} US series" + (f"; failed: {list(us_failed)}" if us_failed else ""))

fetched 16/16 US series
